# Transaction Intelligence — Phase 3: fine-tune the teacher (Colab)

**Before you run:**
1. **Runtime → Change runtime type → GPU** (a free T4 is fine), then Save.
2. If you ran any cells earlier this session, also do **Runtime → Restart session** (resets PyTorch — a torch mismatch is what breaks the imports).

Then **Runtime → Run all**. Bar to beat (TF-IDF baseline): **gold subtype 0.77 / category 0.85**. Cell 4 trains DistilBERT; cell 5 trains the stronger DeBERTa-v3-small. Send both sets of `…/gold` numbers back to Claude.

In [ ]:
# 1) Get the code (idempotent — safe to re-run).
%cd /content
!rm -rf transaction-intelligence
!git clone https://github.com/thejayvaghela/transaction-intelligence.git
%cd transaction-intelligence
# PRIVATE repo? Replace the clone line above with these two:
#   from getpass import getpass; TOKEN = getpass('GitHub token: ')
#   !git clone https://{TOKEN}@github.com/thejayvaghela/transaction-intelligence.git

In [ ]:
# 2) Install deps WITHOUT touching Colab's PyTorch (upgrading torch breaks torchvision).
!pip install -q "transformers>=4.41" datasets accelerate sentencepiece mlflow-skinny
!pip install -q -e . --no-deps

In [ ]:
# 3) Recreate the dataset deterministically (data/ is gitignored; this rebuilds the exact splits).
!python scripts/build_dataset.py

In [ ]:
# 4) Train DistilBERT (~4 epochs, a few min on a T4). Includes temperature-scaling calibration.
#    Prints teacher/test + teacher/gold macro-F1 and a calibrated ECE.
!python scripts/train_transformer.py

In [ ]:
# 5) Train DeBERTa-v3-small (stronger teacher candidate). --no-fp16 (DeBERTa-v3 + fp16 is broken);
#    fp32 is a bit slower (~8-12 min on a T4). Prints teacher-deberta/test + teacher-deberta/gold.
!python scripts/train_transformer.py --model microsoft/deberta-v3-small --output models/teacher-deberta --no-fp16

In [ ]:
# 6) (Optional) Download both trained models as a zip.
!zip -rq teachers.zip models
from google.colab import files
files.download('teachers.zip')

## After training
Paste these four lines back to Claude:
- **`teacher/test`** and **`teacher/gold`** (DistilBERT, calibrated)
- **`teacher-deberta/test`** and **`teacher-deberta/gold`** (DeBERTa-v3-small)

We compare both to the baseline (**gold 0.77 / 0.85**) to choose the teacher for Phase 5 (distillation). Models are in `models/` (cell 6 downloads them).